In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import erf

def relu(x):
    return np.maximum(0, x)

def relu_derivative(x):
    return (x > 0).astype(float)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    s = sigmoid(x)
    return s * (1 - s)

def tanh(x):
    return np.tanh(x)

def tanh_derivative(x):
    return 1 - np.tanh(x)**2

def gelu(x):
    """GELU exact"""
    return 0.5 * x * (1 + erf(x / np.sqrt(2)))

def gelu_derivative(x):
    """Dérivée de GELU exacte: Φ(x) + xφ(x)"""
    phi = (1 / np.sqrt(2 * np.pi)) * np.exp(-0.5 * x**2)   # densité normale
    Phi = 0.5 * (1 + erf(x / np.sqrt(2)))                   # CDF normale
    return Phi + x * phi

def gelu_approx(x):
    """Approximation GELU utilisée en pratique"""
    return 0.5 * x * (1 + np.tanh(np.sqrt(2/np.pi) * (x + 0.044715 * x**3)))

def silu(x):
    """SiLU = Swish avec beta=1"""
    return x * sigmoid(x)

def silu_derivative(x):
    """Dérivée de SiLU"""
    s = sigmoid(x)
    return s + x * s * (1 - s)

# Visualisation
x = np.linspace(-5, 5, 1000)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

activations = [
    ('ReLU', relu, relu_derivative),
    ('Sigmoid', sigmoid, sigmoid_derivative),
    ('Tanh', tanh, tanh_derivative),
    ('GELU', gelu, gelu_derivative),
    ('SiLU/Swish', silu, silu_derivative),
]

for ax, (name, func, deriv) in zip(axes.flat, activations):
    ax.plot(x, func(x), 'b-', label=f'{name}(x)', linewidth=2)
    if deriv is not None:
        ax.plot(x, deriv(x), 'r--', label=f"{name}'(x)", linewidth=2)
    ax.axhline(y=0, color='k', linewidth=0.5)
    ax.axvline(x=0, color='k', linewidth=0.5)
    ax.set_title(name)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('activations_comparison.png', dpi=150)
plt.show()


In [ ]:
import subprocess
subprocess.run(['pip', 'install', 'torch'], check=True)

import torch
import torch.nn as nn
import torch.optim as optim

# ============================================
# PARTIE 1: Tenseurs (= tableaux NumPy + GPU)
# ============================================

# Créer des tenseurs
x = torch.tensor([1.0, 2.0, 3.0])
W = torch.randn(4, 3)  # matrice aléatoire 4×3

print("x:", x)
print("W:", W)
print("W·x:", torch.mv(W, x))  # produit matrice-vecteur

# Calcul automatique des gradients
x = torch.tensor(3.0, requires_grad=True)
y = x**2 + 2*x + 1
y.backward()  # calcule dy/dx automatiquement
print(f"\ndf/dx en x=3: {x.grad}")  # doit donner 8 = 2*3+2

# ============================================
# PARTIE 2: Construire un réseau simple
# ============================================

class MonReseau(nn.Module):
    def __init__(self):
        super().__init__()
        self.couche1 = nn.Linear(2, 8)   # 2 entrées → 8 neurones
        self.couche2 = nn.Linear(8, 4)   # 8 → 4
        self.couche3 = nn.Linear(4, 1)   # 4 → 1 sortie
        self.relu = nn.ReLU()
        self.gelu = nn.GELU()  # LA fonction de ta thèse!
    
    def forward(self, x):
        x = self.relu(self.couche1(x))   # ReLU sur couche 1
        x = self.gelu(self.couche2(x))   # GELU sur couche 2 (thèse!)
        x = self.couche3(x)
        return x

reseau = MonReseau()
print("\nArchitecture du réseau:")
print(reseau)

# Test forward pass
x_test = torch.randn(1, 2)  # 1 exemple avec 2 features
sortie = reseau(x_test)
print(f"\nEntrée: {x_test}")
print(f"Sortie: {sortie}")

# ============================================
# PARTIE 3: Entraîner le réseau
# ============================================

# Données XOR (non-linéaire, nécessite réseau profond)
X_train = torch.tensor([[0.,0.], [0.,1.], [1.,0.], [1.,1.]])
y_train = torch.tensor([[0.], [1.], [1.], [0.]])  # XOR

# Optimiseur et perte
optimizer = optim.Adam(reseau.parameters(), lr=0.01)
criterion = nn.BCEWithLogitsLoss()

# Boucle d'entraînement
for epoch in range(500):
    optimizer.zero_grad()          # Remettre gradients à 0
    sortie = reseau(X_train)       # Forward pass
    perte = criterion(sortie, y_train)  # Calculer erreur
    perte.backward()               # Backpropagation
    optimizer.step()               # Mettre à jour poids
    
    if epoch % 100 == 0:
        print(f"Epoch {epoch}: Loss = {perte.item():.4f}")

# Test final
with torch.no_grad():
    predictions = torch.sigmoid(reseau(X_train))
    print("\nPrédictions XOR:")
    for xi, yi, pi in zip(X_train, y_train, predictions):
        print(f"  {xi.numpy()} → vrai: {yi.item():.0f}, prédit: {pi.item():.3f}")


ModuleNotFoundError: No module named 'torch'